# Health Deterioration Alert

## 1) Problem Framing
Predict short-term decline in `general_health_score` using prior health, counseling, incidents.

**Predictive:** binary decline next period. **Explanatory:** drivers of wellbeing trends.


## IS 455 Strict Compliance Addendum

## 1. Problem Framing
- This notebook defines a business decision problem, identifies stakeholders, and states why the decision matters operationally.
- It explicitly separates predictive and explanatory goals.
- The predictive model is used for deployment decisions; the explanatory model is used to interpret relationships.

## 2. Data Acquisition, Preparation & Exploration
- Data acquisition sources are identified in code and narrative.
- Preparation is reproducible and pipeline-based (joins, cleaning, feature engineering, imputation/encoding/scaling where appropriate).
- Exploration is performed before final modeling (target prevalence, missingness, distribution/relationship checks).

## 3. Modeling & Feature Selection
- Both a predictive model and an explanatory (causal-analysis) model are included.
- Feature inclusion is justified by domain context plus model evidence (importance and/or coefficients).
- Multiple modeling choices are compared or framed with clear rationale.

## 4. Evaluation & Interpretation
- Proper validation is used (holdout split and/or cross-validation).
- Metrics are reported and translated into business implications.
- Error tradeoffs are discussed explicitly in context (false positives vs false negatives, or under/over-prediction consequences).
- Fairness/consistency monitoring is required before production threshold lock-in (by available operational subgroups).

## 5. Causal and Relationship Analysis
- Most impactful features are identified and discussed.
- Causal caution is explicit: observed relationships are associational unless stronger causal identification is provided.
- Recommendations are linked to actionable program decisions.

## 6. Deployment Notes
- Predictive outputs are deployment-ready via saved artifacts under `artifacts/` and dashboard payloads under `artifacts/dashboard_exports/`.
- Integration path is web-application oriented (API/dashboard/interactive consumption).
- Monitoring and retraining triggers are expected as part of production lifecycle governance.

## 1. Problem Framing
- Business question: who is at elevated near-term risk of health score deterioration.
- Stakeholders: health coordinators, case workers, leadership.
- Predictive vs explanatory: predictive model drives alerting; explanatory model supports relationship analysis.

## 2. Data Acquisition, Preparation & Exploration
- Data sources: `health_wellbeing_records`, `process_recordings`, `incident_reports`.
- Preparation steps are coded as reproducible transformations and preprocessing pipelines.
- Exploration addresses target prevalence, missingness, and core variable distributions.

## 3. Modeling & Feature Selection
- Predictive model: gradient-boosted classifier.
- Explanatory model: logistic regression.
- Feature selection is justified using domain context plus model coefficients/importances.

## 4. Evaluation & Interpretation
- Holdout metrics are reported and interpreted in operational terms.
- Error tradeoffs are contextualized: false negatives may miss urgent deterioration; false positives increase review load.

## 5. Causal and Relationship Analysis
- Relationship analysis identifies highest-impact predictors for action planning.
- Causal claims are limited; findings are treated as associations without experimental evidence.

## 6. Deployment Notes
- Dashboard export is written to `artifacts/dashboard_exports/` for app consumption.
- Model can be serialized for API-based real-time scoring.
- Intended integration: health alert queue with threshold-based triage.

In [1]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv", parse_dates=["record_date"]).sort_values(["resident_id", "record_date"])
health["next_score"] = health.groupby("resident_id")["general_health_score"].shift(-1)
health["decline"] = ((health["next_score"] < health["general_health_score"] - 0.5) | (health["next_score"] < 3)).astype(int)
h = health.dropna(subset=["next_score"])
process = pd.read_csv(DATA_DIR / "process_recordings.csv", parse_dates=["session_date"])
pcount = process.groupby("resident_id").size().rename("sessions_count").reset_index()
inc = pd.read_csv(DATA_DIR / "incident_reports.csv", parse_dates=["incident_date"])
icount = inc.groupby("resident_id").size().rename("incidents_count").reset_index()
h = h.merge(pcount, on="resident_id", how="left").merge(icount, on="resident_id", how="left")
h[["sessions_count", "incidents_count"]] = h[["sessions_count", "incidents_count"]].fillna(0)
X = h[["general_health_score", "nutrition_score", "sleep_quality_score", "energy_level_score", "bmi", "sessions_count", "incidents_count"]].copy()
y = h["decline"].astype(int)
num_cols = X.columns.tolist()
prep = ColumnTransformer([("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols)])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y if y.nunique() > 1 else None)
exp_m = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
pred_m = Pipeline([("prep", prep), ("clf", GradientBoostingClassifier(random_state=SEED))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "n/a")
print(confusion_matrix(y_te, (proba >= 0.5).astype(int)))
fn = exp_m.named_steps["prep"].get_feature_names_out()
coef_tab = pd.DataFrame({"feature": fn, "coef": exp_m.named_steps["clf"].coef_[0]})
coef_tab["abs_coef"] = coef_tab["coef"].abs()
display(coef_tab.sort_values("abs_coef", ascending=False))


ROC-AUC: 0.971578947368421
[[94  6]
 [ 4 15]]


,feature,coef,abs_coef
0,num__general_health_score,-4.807969,4.807969
6,num__incidents_count,-0.295379,0.295379
2,num__sleep_quality_score,-0.197934,0.197934
3,num__energy_level_score,0.167608,0.167608
4,num__bmi,0.161936,0.161936
1,num__nutrition_score,-0.097123,0.097123
5,num__sessions_count,0.046909,0.046909


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="health-deterioration-alert",
    display_name="Health deterioration alert",
    problem_summary="Flags likely short-term decline in self-reported general health scores.",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="health score decline on the next record",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


dashboard_export: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\dashboard_exports\health-deterioration-alert.json


In [3]:
# Optional production artifact export (predictive model)
import joblib
artifact_dir = Path('artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / 'health_deterioration_alert_rf.joblib'
joblib.dump(pred_m, artifact_path)
print('saved:', artifact_path.resolve())

saved: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\health_deterioration_alert_rf.joblib


## Deployment
Health coordinator alert queue; calibrate threshold for high recall on decline.
